<h2>Exercise 03. Aggregations</h2>

In [2]:
import pandas as pd 
import sqlite3

### 1. Create a connection to the database using the sqlite3 library.

In [3]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

### 2. Get the schema of the test table.

In [4]:
test = pd.io.sql.read_sql(
    '''
    SELECT * FROM test LIMIT 10''',
    conn
)

test

,uid,labname,first_commit_ts,first_view_ts
0,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 10:56:55.833899
1,user_30,laba04,2020-04-18 13:36:53.971502,2020-04-17 22:46:26.785035
2,user_30,laba04s,2020-04-18 14:51:37.498399,2020-04-17 22:46:26.785035
3,user_14,laba04,2020-04-18 15:14:00.312338,2020-04-18 10:53:52.623447
4,user_14,laba04s,2020-04-18 22:30:30.247628,2020-04-18 10:53:52.623447
5,user_19,laba04,2020-04-20 19:05:01.297780,2020-04-21 20:30:38.034966
6,user_25,laba04,2020-04-20 19:16:50.673054,2020-05-09 23:54:54.260791
7,user_21,laba04,2020-04-21 17:48:00.487806,2020-04-22 22:40:36.824081
8,user_30,project1,2020-04-22 12:36:24.053518,2020-04-17 22:46:26.785035
9,user_21,laba04s,2020-04-22 20:09:21.857747,2020-04-22 22:40:36.824081


### 4. Find the minimum value of the delta between the first commit and the deadline of the corresponding lab for all users using only one query.

In [48]:
df_min = pd.io.sql.read_sql(
    '''
    SELECT t.uid, t.labname, t.first_commit_ts, t.first_view_ts, d.deadlines,
    MIN((strftime('%s', t.first_commit_ts)-d.deadlines) / 3600.0) as min_delta
    FROM test t
    INNER JOIN deadlines d
        ON t.labname = d.labs
    WHERE t.labname <> 'project1'
    GROUP BY t.uid
    ''',
    conn
)

df_min

,uid,labname,first_commit_ts,first_view_ts,deadlines,min_delta
0,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136,1590364799,-175.556667
1,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714,1590364799,-132.341944
2,user_14,laba04,2020-04-18 15:14:00.312338,2020-04-18 10:53:52.623447,1587945599,-200.766389
3,user_17,laba04,2020-04-23 14:24:29.947554,2020-04-18 10:56:55.833899,1587945599,-81.591667
4,user_18,laba05,2020-05-03 13:01:34.848756,2020-04-26 22:49:29.243278,1588550399,-10.973611
5,user_19,laba04,2020-04-20 19:05:01.297780,2020-04-21 20:30:38.034966,1587945599,-148.916111
6,user_21,laba04,2020-04-21 17:48:00.487806,2020-04-22 22:40:36.824081,1587945599,-126.199722
7,user_25,laba06,2020-05-18 17:07:47.988807,2020-05-09 23:54:54.260791,1590364799,-150.870000
8,user_28,laba06,2020-05-17 17:08:48.257050,2020-05-10 21:07:50.350946,1590364799,-174.853056
9,user_3,laba06,2020-05-17 09:56:40.480319,2020-05-08 10:53:47.123832,1590364799,-182.055278


In [40]:
df_max = pd.io.sql.read_sql(
    '''
    SELECT t.uid, t.labname, t.first_commit_ts, t.first_view_ts, d.deadlines,
    MAX((strftime('%s', t.first_commit_ts)-d.deadlines) / 3600.0) as max_delta
    FROM test t
    INNER JOIN deadlines d
    ON t.labname = d.labs
    WHERE t.labname <> 'project1'
    GROUP BY t.uid
    ''',
    conn
)

df_max

,uid,labname,first_commit_ts,first_view_ts,deadlines,max_delta
0,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136,1587945599,-6.796667
1,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714,1587945599,-39.368056
2,user_14,laba05,2020-04-30 11:33:04.523118,2020-04-18 10:53:52.623447,1588550399,-84.448611
3,user_17,laba05,2020-05-02 13:21:24.045876,2020-04-18 10:56:55.833899,1588550399,-34.643056
4,user_18,laba04s,2020-04-26 20:03:56.935458,2020-04-26 22:49:29.243278,1587945599,-3.934167
5,user_19,laba05,2020-05-02 15:16:13.586405,2020-04-21 20:30:38.034966,1588550399,-32.729444
6,user_21,laba05,2020-05-02 14:05:40.013959,2020-04-22 22:40:36.824081,1588550399,-33.905278
7,user_25,laba04s,2020-04-26 21:07:56.952117,2020-05-09 23:54:54.260791,1587945599,-2.867500
8,user_28,laba04s,2020-04-26 15:53:44.906136,2020-05-10 21:07:50.350946,1587945599,-8.104167
9,user_3,laba05,2020-05-01 11:29:17.988118,2020-05-08 10:53:47.123832,1588550399,-60.511667


In [41]:
df_avg = pd.io.sql.read_sql(
    '''
    SELECT  t.labname, t.first_commit_ts, t.first_view_ts, d.deadlines,
    AVG((strftime('%s', t.first_commit_ts)-d.deadlines) / 3600.0) as avg_delta
    FROM test t
    INNER JOIN deadlines d
    ON t.labname = d.labs
    WHERE t.labname <> 'project1'
    GROUP BY t.uid
    ''',
    conn
)

df_avg

,labname,first_commit_ts,first_view_ts,deadlines,avg_delta
0,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136,1587945599,-65.119778
1,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714,1587945599,-75.242444
2,laba04,2020-04-18 15:14:00.312338,2020-04-18 10:53:52.623447,1587945599,-159.568796
3,laba04,2020-04-23 14:24:29.947554,2020-04-18 10:56:55.833899,1587945599,-62.207667
4,laba04,2020-04-26 19:48:11.822365,2020-04-26 22:49:29.243278,1587945599,-6.368148
5,laba04,2020-04-20 19:05:01.297780,2020-04-21 20:30:38.034966,1587945599,-99.440417
6,laba04,2020-04-21 17:48:00.487806,2020-04-22 22:40:36.824081,1587945599,-96.111181
7,laba04,2020-04-20 19:16:50.673054,2020-05-09 23:54:54.260791,1587945599,-93.474944
8,laba04,2020-04-22 21:47:19.707242,2020-05-10 21:07:50.350946,1587945599,-86.793833
9,laba04,2020-04-23 20:29:14.054364,2020-05-08 10:53:47.123832,1587945599,-105.738222


In [57]:
views_diff = pd.io.sql.read_sql(
    '''
    SELECT t.uid, 
    AVG(
        (strftime('%s', t.first_commit_ts) - d.deadlines) / 3600.0
    ) AS avg_delta,
    COUNT(DISTINCT p.datetime) AS pageviews
    FROM test t
    JOIN deadlines d
        ON t.labname = d.labs
    LEFT JOIN pageviews p
        ON t.uid = p.uid
    WHERE t.labname <> 'project1'
    GROUP BY t.uid
    ''',
    conn
)

views_diff


,uid,avg_delta,pageviews
0,user_1,-65.119778,28
1,user_10,-75.242444,89
2,user_14,-159.568796,143
3,user_17,-62.207667,47
4,user_18,-6.368148,3
5,user_19,-99.440417,16
6,user_21,-96.111181,10
7,user_25,-93.474944,179
8,user_28,-86.793833,149
9,user_3,-105.738222,317


In [58]:
views_diff.corr(numeric_only=True)

,avg_delta,pageviews
avg_delta,1.000000,-0.279143
pageviews,-0.279143,1.000000


In [59]:
conn.close()